# Machine-Learning Emulators (aemu) — documentation examples

Companion notebook to the **Machine-Learning Emulators (aemu)** documentation
page (`docs/source/aemu_integration.rst`). One section per code snippet /
figure; figure sections regenerate the `tpayne_*` / `feh_*` images in
`docs/img/` (light and dark variants).

Requires the `aemu` extra (`pip install "stellar-spice[aemu]"`); downloading
a pretrained bundle needs network access, so this notebook is shipped
unexecuted.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("JAX_ENABLE_X64", "1")   # TransformerPayne bundles need 64-bit JAX

from pathlib import Path
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm

DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Render make_fig() in light and dark styles into docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")


## Load a pretrained intensity bundle

The name may be a Hugging Face repo id or a local bundle directory.

In [2]:
from spice.spectrum import IntensityPretrainedAemuSpectrumEmulator

# The emulator of the SPICE paper, pinned to its v1.2.0 release
emu = IntensityPretrainedAemuSpectrumEmulator("mjablonska/TPayne-spice-harps-interp",
                                              revision="v1.2.0")
print(emu.wavelength_range)
print(emu.stellar_parameter_names)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

(3780.000000000002, 6910.000000000002)
['teff', 'logg', 'feh', 'vmicro', 'a', 'c', 'n', 'o', 'r', 's']


## Spectrum of a rotating star

Figure: `tpayne_spectrum_rotation.png` / `tpayne_spectrum_rotation_dark.png`

In [3]:
from spice.models import IcosphereModel
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import simulate_observed_flux

m = IcosphereModel.construct(5000, 1., 1.,   # dense enough for 100 km/s at R ~ 5e5 (see Appendix E of the paper)
                             emu.to_parameters(dict(teff=7000., logg=4.3)),
                             emu.stellar_parameter_names)

mt = evaluate_rotation(add_rotation(m, 100, jnp.array([0., 1., 0.])), 0.)

vws = np.linspace(5000, 5020, 2000)   # the bundle covers 3780-6910 A
spec_no_rot = simulate_observed_flux(emu.intensity, m, jnp.log10(vws))
spec_rot = simulate_observed_flux(emu.intensity, mt, jnp.log10(vws))

def rotation_spectrum_fig():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(vws, spec_no_rot[:, 0], color='black', linewidth=1, label='No rotation')
    ax.plot(vws, spec_rot[:, 0], color='royalblue', linewidth=3, label='100 km/s')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Flux [erg/s/cm$^2$/$\AA$]')
    ax.legend()
    return fig

save_doc_fig("tpayne_spectrum_rotation", rotation_spectrum_fig)

PHOEBE: passband "Bolometric:900-40000" has a newer version available.  Run phoebe.list_passband_online_history("Bolometric:900-40000") to get a list of available changes and phoebe.update_passband("Bolometric:900-40000") or phoebe.update_all_passbands() to update.


[spice] IcosphereModel constructed in 0.9 s


saved tpayne_spectrum_rotation.png / tpayne_spectrum_rotation_dark.png


## A rotating star with an iron spot

Figures: `feh_spot_0.png`, `feh_spot_50.png` (+`_dark`)

In [4]:
from spice.models.spots import add_spot
from spice.plots import plot_3D

timestamps = np.linspace(0, 48*3600, 50)

feh_index = emu.stellar_parameter_names.index('feh')
m_spotted = add_spot(m, spot_center_theta=1., spot_center_phi=1., spot_radius=30.,
                     parameter_delta=0.5, parameter_index=feh_index)
m_spotted = [evaluate_rotation(add_rotation(m_spotted, 25.), t) for t in timestamps]

save_doc_fig("feh_spot_0",
             lambda: plot_3D(m_spotted[0], property_label='[Fe/H]', property=feh_index)[0])
save_doc_fig("feh_spot_50",
             lambda: plot_3D(m_spotted[25], property_label='[Fe/H]', property=feh_index)[0])   # t = 24 h

saved feh_spot_0.png / feh_spot_0_dark.png


saved feh_spot_50.png / feh_spot_50_dark.png


## Line profiles across the rotation

Figure: `feh_line_profile.png` / `feh_line_profile_dark.png`

In [5]:
vws_line = np.linspace(5004.5, 5007.5, 400)   # Fe I 5005.71 + 5006.12 A
spec_rot_spotted = [simulate_observed_flux(emu.intensity, _m, jnp.log10(vws_line))
                    for _m in tqdm(m_spotted)]

def line_profile_fig():
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.cool(np.linspace(0, 1, len(spec_rot_spotted)))
    for i, spectrum in enumerate(spec_rot_spotted):
        ax.plot(vws_line, spectrum[:, 0], color=colors[i], linewidth=1, alpha=0.5)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.cool,
                               norm=plt.Normalize(vmin=0, vmax=timestamps[-1]/3600))
    fig.colorbar(sm, ax=ax, label='Time [h]')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Flux [erg/s/cm$^2$/$\AA$]')
    return fig

save_doc_fig("feh_line_profile", line_profile_fig)

  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:00<00:31,  1.55it/s]

  4%|▍         | 2/50 [00:19<09:02, 11.30s/it]

  6%|▌         | 3/50 [00:38<11:30, 14.68s/it]

  8%|▊         | 4/50 [00:55<11:59, 15.65s/it]

 10%|█         | 5/50 [01:11<11:59, 15.99s/it]

... (43 outputs left out)

 98%|█████████▊| 49/50 [16:23<00:17, 17.81s/it]

100%|██████████| 50/50 [16:41<00:00, 17.97s/it]

100%|██████████| 50/50 [16:41<00:00, 20.03s/it]

saved feh_line_profile.png / feh_line_profile_dark.png


## Line profiles of a pulsating star

Figure: `tpayne_pulsation.png` / `tpayne_pulsation_dark.png`

In [6]:
from spice.models.mesh_transform import add_pulsation, evaluate_pulsations
import cmasher as cmr

# radial mode, amplitude in solar radii, period in days
mp = add_pulsation(m, 0, 0, 0.1, jnp.array([[0.03, 0.]]))

TIMESTAMPS = jnp.linspace(0., 0.1, 20)   # one period, in days
mps = [evaluate_pulsations(mp, t) for t in tqdm(TIMESTAMPS)]
specs = [simulate_observed_flux(emu.intensity, _m, jnp.log10(vws_line)) for _m in tqdm(mps)]

v_max = max(float(jnp.max(jnp.abs(_m.los_velocities))) for _m in mps)
print(f"peak line-of-sight pulsation velocity: {v_max:.1f} km/s "
      f"(2 pi A / P = {2 * np.pi * 0.03 * 695700. / (0.1 * 86400.):.1f} km/s)")


def pulsation_profiles_fig():
    cmap = cmr.bubblegum
    norm = plt.Normalize(TIMESTAMPS.min(), TIMESTAMPS.max())
    fig, ax = plt.subplots(figsize=(9, 5))
    for spec, timestamp in zip(specs, TIMESTAMPS):
        ax.plot(vws_line, spec[:, 0] / spec[:, 1], color=cmap(norm(timestamp)))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('Time [d]')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel('Normalized flux')
    return fig

save_doc_fig("tpayne_pulsation", pulsation_profiles_fig)

  0%|          | 0/20 [00:00<?, ?it/s]

[spice] Pulsations evaluated in 3.1 s
  0%|          | 0/20 [00:03<?, ?it/s]

  5%|▌         | 1/20 [00:03<01:03,  3.32s/it]

[spice] Pulsations evaluated in 1.6 s
  5%|▌         | 1/20 [00:04<01:03,  3.32s/it]

 10%|█         | 2/20 [00:05<00:45,  2.51s/it]

[spice] Pulsations evaluated in 1.7 s
 10%|█         | 2/20 [00:06<00:45,  2.51s/it]

... (56 outputs left out)

100%|██████████| 20/20 [05:38<00:00, 17.81s/it]

100%|██████████| 20/20 [05:39<00:00, 16.95s/it]

peak line-of-sight pulsation velocity: 15.1 km/s (2 pi A / P = 15.2 km/s)


saved tpayne_pulsation.png / tpayne_pulsation_dark.png
